# Validation of electronic health records agreement with self-reported

## Purpose
This notebook evaluates agreement between self-reported (SR) and electronic health record (EHR)–derived diagnoses for selected chronic conditions in the Our Future Health (OFH) cohort. Binary indicators for each condition are derived from questionnaire responses and linked NHS England inpatient (HES) data, and concordance is assessed using prevalence estimates and Cohen’s kappa with bootstrap confidence intervals.

## Outputs
- Participant-level binary indicators for SR- and EHR-derived conditions
- Aggregated counts and prevalence estimates for:
    - SR only
    EHR only
    Both SR and EHR
    Either source
- Cohen’s kappa statistics with 95% confidence intervals for each condition
- Summary tables used for reporting agreement results in the manuscript and supplementary materials

## Relationship to manuscript
Outputs from this notebook are used to generate **Extended Data Figures 8** in the main text and to populate **Supplementary Table 16** (*Agreement between self-reported and EHR-derived chronic conditions in Our Future Health*).

## Data and access notes
Analyses use restricted Our Future Health data accessed within the OFH Trusted Research Environment under approved study permissions. All outputs are aggregated, non-disclosive summary statistics and comply with OFH Safe Output requirements.

## Notes
Analyses are restricted to participants with both SR and linked inpatient EHR data. EHR diagnoses are defined using ICD-10 codes applied with prefix matching (codes ≤3 characters capture all subcodes) and are considered present if recorded in any diagnosis position across hospital admissions. SR conditions are based on responses to “ever diagnosed by a doctor or health professional” questionnaire items. Differences between SR and EHR estimates reflect variation in ascertainment (e.g., recall, clinical coding practices, and inpatient vs community care capture). ICD-10 code mappings used to define EHR conditions are provided in the Supplementary Information Table 3.

### Setup

In [ ]:
# Download phenofhy
!dx download profile_study_v4:/applets/phenofhy -r

# Import packages
import dxpy
import shlex
import subprocess
import numpy as np
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

# Import phenofhy
import phenofhy

### Initialize Spark

spark = SparkSession.builder \
    .appName("Phenotype Analysis") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.kryoserializer.buffer.max", "128") \
    .getOrCreate()

### Load fields

In [ ]:
files = [
    "ehr_nhse_inpat_diag_icd_fields.csv",
]

phenofhy.utils.download_files([
    (str(phenofhy.utils.find_latest_dx_file_id(f)), f"inputs/{f}")
    for f in files
])

pheno_dfs = {f.replace('.csv', ''): pd.read_csv(f'./inputs/{f}') for f in files}

metadata_dfs = phenofhy.load.metadata()

### Extraction

In [ ]:
phenofhy.load.field_list(
    input_file="inputs/ehr_nhse_inpat_diag_icd_fields.csv", 
    output_file="outputs/intermediate/ehr_nhse_inpat_diag_fields_metadata.csv",
)

phenofhy.extract.fields(
    input_file="outputs/intermediate/ehr_nhse_inpat_diag_fields_metadata.csv",
    output_file="outputs/raw/ehr_nhse_inpat_diag_fields_raw_values_query.sql", 
    cohort_key="FULL_SAMPLE_ID", 
    sql_only=True
)

raw_nhse_inpat_df = phenofhy.extract.sql_to_pandas(
    "outputs/raw/ehr_nhse_inpat_diag_fields_raw_values_query.sql"
)

### Processing

In [ ]:
# Self-report side — one row per participant
sr_raw_df = raw_nhse_inpat_df[['participant.pid', 
            'questionnaire.diag_2_m',
            'questionnaire.diag_resp_1_m',
            'questionnaire.diag_endocr_1_m',
            'questionnaire.diag_psych_1_m',
            'questionnaire.diag_cvd_1_m',
            'questionnaire.diag_osteo_1_m']].drop_duplicates('participant.pid')

sr_intermed_df = phenofhy.process.participant_fields(sr_raw_df)
sr_df = phenofhy.process.questionnaire_fields(sr_intermed_df, derive=False)

# EHR side — one row per admission (multiple rows per participant)
ehr_df = raw_nhse_inpat_df[['nhse_eng_inpat.pid'] + 
            [c for c in raw_nhse_inpat_df.columns if 'nhse_eng_inpat.diag_4_' in c]]

# Identify SR 

def has_code(cell, code):
    """Check if a list-type cell contains a specific code."""
    if cell is None:
        return 0
    return int(code in cell)

# SR binary flags
sr_df['sr_cancer']          = sr_df['questionnaire.diag_2_m'].apply(lambda x: has_code(x, 3))
sr_df['sr_asthma']          = sr_df['questionnaire.diag_resp_1_m'].apply(lambda x: has_code(x, 4))
sr_df['sr_diabetes']        = sr_df['questionnaire.diag_endocr_1_m'].apply(lambda x: has_code(x, 2))
sr_df['sr_depression']      = sr_df['questionnaire.diag_psych_1_m'].apply(lambda x: has_code(x, 4))
sr_df['sr_hypertension']    = sr_df['questionnaire.diag_cvd_1_m'].apply(lambda x: has_code(x, 9))
sr_df['sr_cholesterol']     = sr_df['questionnaire.diag_cvd_1_m'].apply(lambda x: has_code(x, 4))
sr_df['sr_osteoarthritis']  = sr_df['questionnaire.diag_osteo_1_m'].apply(lambda x: has_code(x, 3))

from phenofhy import icd

# Identify ICD matches
trait_map = {
    'ehr_cancer':         ['C'],
    'ehr_asthma':         ['J45', 'J46'],
    'ehr_diabetes':       ['E11'],
    'ehr_depression':     ['F32', 'F33'],
    'ehr_hypertension':   ['I10'],
    'ehr_cholesterol':    ['E78'],
    'ehr_osteoarthritis': ['M15', 'M16', 'M17', 'M18', 'M19'],
}

trait_pids, summary = icd.match_icd_traits(ehr_df, trait_map, primary_only=False)

# Get binary columns
all_pids = ehr_df['nhse_eng_inpat.pid'].unique()
ehr_binary = pd.DataFrame({'participant.pid': all_pids})

for condition, pids in trait_pids.items():
    ehr_binary[condition] = ehr_binary['participant.pid'].isin(pids).astype(int)

# Merge
combined_df = sr_df[['participant.pid', 'sr_cancer', 'sr_asthma', 'sr_diabetes',
                      'sr_depression', 'sr_hypertension', 'sr_cholesterol', 
                      'sr_osteoarthritis']].merge(
    ehr_binary,
    on='participant.pid',
    how='inner'  # only participants with both SR and EHR data
)

### Analysis

In [ ]:
import pandas as pd
import numpy as np

conditions = {
    'Cancer':           ('sr_cancer',        'ehr_cancer'),
    'Asthma':           ('sr_asthma',        'ehr_asthma'),
    'Diabetes':         ('sr_diabetes',       'ehr_diabetes'),
    'Depression':       ('sr_depression',     'ehr_depression'),
    'Hypertension':     ('sr_hypertension',   'ehr_hypertension'),
    'High Cholesterol': ('sr_cholesterol',    'ehr_cholesterol'),
    'Osteoarthritis':   ('sr_osteoarthritis', 'ehr_osteoarthritis'),
}

def cohen_kappa(sr, ehr):
    n = len(sr)
    p_o = (sr == ehr).sum() / n
    p_sr_pos = sr.sum() / n
    p_sr_neg = 1 - p_sr_pos
    p_ehr_pos = ehr.sum() / n
    p_ehr_neg = 1 - p_ehr_pos
    p_e = (p_sr_pos * p_ehr_pos) + (p_sr_neg * p_ehr_neg)
    return (p_o - p_e) / (1 - p_e)

def bootstrap_kappa(sr, ehr, n_boot=1000, ci=0.95):
    kappas = []
    n = len(sr)
    for _ in range(n_boot):
        idx = np.random.randint(0, n, n)
        kappas.append(cohen_kappa(sr[idx], ehr[idx]))
    lower = np.percentile(kappas, (1 - ci) / 2 * 100)
    upper = np.percentile(kappas, (1 + ci) / 2 * 100)
    return lower, upper

rows = []
for condition, (sr_col, ehr_col) in conditions.items():
    sr  = combined_df[sr_col].values
    ehr = combined_df[ehr_col].values

    n_validated = len(sr)
    n_both      = int(((sr == 1) & (ehr == 1)).sum())
    n_sr_only   = int(((sr == 1) & (ehr == 0)).sum())
    n_ehr_only  = int(((sr == 0) & (ehr == 1)).sum())
    n_either    = int(((sr == 1) | (ehr == 1)).sum())

    prev_both     = n_both     / n_validated
    prev_sr_only  = n_sr_only  / n_validated
    prev_ehr_only = n_ehr_only / n_validated
    prev_either   = n_either   / n_validated

    kappa            = cohen_kappa(sr, ehr)
    kappa_lo, kappa_hi = bootstrap_kappa(sr, ehr)

    rows.append({
        'condition':    condition,
        'n_validated':  n_validated,
        'n_both':       n_both,
        'n_sr_only':    n_sr_only,
        'n_ehr_only':   n_ehr_only,
        'n_either':     n_either,
        'prev_both':    round(prev_both     * 100, 1),
        'prev_sr_only': round(prev_sr_only  * 100, 1),
        'prev_ehr_only':round(prev_ehr_only * 100, 1),
        'prev_either':  round(prev_either   * 100, 1),
        'kappa':        round(kappa, 3),
        'kappa_lower':  round(kappa_lo, 3),
        'kappa_upper':  round(kappa_hi, 3),
    })

agreement_df = pd.DataFrame(rows)
print(agreement_df)

### Sample size check

In [ ]:
n_sr_total = sr_df['participant.pid'].nunique()
n_ehr_total = ehr_binary['participant.pid'].nunique()
n_both = combined_df['participant.pid'].nunique()

print(n_sr_total, n_ehr_total, n_both)